# 🚗 Lesson T1 — Turn the CLI Drafter into a Web Service

**The seed becomes a car. By the end of this notebook, anyone at PacificFreight can call the drafter over HTTP — no laptop required.**

Time: 30 min read + 30 min run. Topic: FastAPI, Pydantic, REST endpoints, Docker.

---

## 🎯 What you will build

```
  Before (Phase 1):                       After (Phase 2):
  ─────────────────                       ───────────────
                                          
  📧 Mei types an email                   📧 Anyone at PacificFreight
  in her terminal                         opens a browser, calls a URL
       │                                         │
       ▼                                         ▼
  ┌─────────────┐                       ┌──────────────────────┐
  │  Mei's      │                       │  🌐 The drafter is  │
  │  laptop     │                       │  now a web service  │
  │             │                       │  on Daniel's VM     │
  │  $ python3  │                       │                      │
  │    drafter  │                       │  GET  /health        │
  │             │                       │  POST /draft         │
  │  → draft    │                       │  POST /retrieve      │
  └─────────────┘                       │  POST /eval          │
                                       │                      │
  😟 Only Mei can use it.               │  → draft             │
                                       └──────────────────────┘
                                       
                                       🎉 The whole company can use it!
```

**The big idea:** the drafter is the SAME drafter. We just wrapped it in a web server. The 5 pieces (READ, EXTRACT, LOOKUP, DRAFT, OUTPUT) didn't change. We just added a 6th piece on the front: a **door** (the HTTP endpoint) that lets other programs knock on it.

## 🧠 Concept (5 min) — What is a web service, really?

A **web service** is a program that:
1. Listens on a **port** (like a phone number — 8000 is a popular one)
2. Speaks **HTTP** (the language of the internet)
3. Responds to **requests** from other programs

Think of it like a restaurant:
```
  🍽️  RESTAURANT ANALOGY
  ══════════════════════

  You (the customer)     →  Another program (a browser, a script, another service)
  The menu               →  The API docs (what URLs you can call, what data to send)
  You order food         →  You make an HTTP request (GET, POST, etc.)
  The kitchen cooks      →  The service runs the drafter pipeline
  The waiter brings food →  The service sends back an HTTP response (the draft)
  The restaurant         →  The FastAPI app
  The waiter             →  uvicorn (the server program)
```

When a customer's email arrives, here's what happens now:
```
  📧 Email lands in Mei's inbox
        │
        ▼
  🌐 Mei's email client sends HTTP POST to the drafter service:
        POST http://daniels-vm:8000/draft
        Body: {"email": "Where is my parcel PF-1003?"}
        │
        ▼
  🤖 The drafter service runs the 5-piece pipeline
        │
        ▼
  ✉️  Service sends HTTP response back:
        {"draft": "Hi! Your parcel PF-1003 is in customs..."}
        │
        ▼
  📋 Mei's email client puts the draft in her reply box
```

**Why this matters:** Mei no longer needs to leave her email client. The drafter is just one button-click away. And Sarah (ops manager) can call the same drafter from her own dashboard. And the analytics team can call it from a script. **One drafter, many callers.**

## 🏗️ Build it

We'll wrap the Phase 1 drafter in FastAPI. FastAPI is a Python library that makes it easy to build web services.

In [ ]:
# Step 0 — imports
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import Optional
import json
import re
from pathlib import Path
from dataclasses import dataclass

print("✅ All imports OK")
print("   FastAPI is the web framework.")
print("   Pydantic validates the input (catches typos before they break things).")
print("   uvicorn is the server that runs FastAPI.")


### Step 1 — Define the request and response shapes

Before we build the service, we tell FastAPI: "this is what the request looks like, and this is what the response looks like." This is called the **schema** — like a form that says "name: text, age: number."

Pydantic does this for us. It also **validates** the input: if a user sends `{"email": 12345}` (a number instead of text), Pydantic catches it and returns a helpful error before anything bad happens.

In [ ]:
# Step 1 — define request/response shapes with Pydantic
class DraftRequest(BaseModel):
    """The shape of a POST /draft request.

    Field() lets us add constraints:
    - min_length: must be at least 3 characters (no empty emails)
    - max_length: must be at most 5000 characters (no novels)
    """
    email: str = Field(..., min_length=3, max_length=5000, description="The customer's email text")
    shipment_id: Optional[str] = Field(None, description="Optional shipment ID if known")

class DraftResponse(BaseModel):
    """The shape of a POST /draft response."""
    shipment_id: Optional[str] = None
    draft: str
    sources: list[str] = []
    model: str = "mock"
    cost_usd: float = 0.0

# Try to validate some bad input (this would crash in Phase 1)
try:
    bad = DraftRequest(email="")  # too short
except Exception as e:
    print(f"✅ Pydantic caught the bad input: {type(e).__name__}")
    print(f"   {str(e)[:200]}")

### Step 2 — Wrap the 5-piece pipeline in a FastAPI app

Now the magic. We take the Phase 1 drafter (the 5 pieces) and put them inside a FastAPI app. Each piece becomes an **endpoint** (a URL someone can call).

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import Optional
import json
import re
from pathlib import Path
from dataclasses import dataclass

# Find phase-1-foundations/shared/ by searching up + sibling paths
PHASE1 = None
for base in [Path.cwd(), *Path.cwd().parents]:
    for candidate in [base / "shared" / "shipments.json",
                      base / "course" / "ai-fde" / "phase-1-foundations" / "shared" / "shipments.json"]:
        if candidate.exists():
            PHASE1 = candidate.parent.parent
            break
    if PHASE1: break
if PHASE1 is None:
    raise FileNotFoundError("Could not find phase-1-foundations/shared/shipments.json")

print(f"OK  PHASE1 = {PHASE1}")


### Step 3 — Test the service in-process

FastAPI has a built-in test client so we can call the endpoints without running a real server. This is how Phase 2's 13 pytest cases work.

In [ ]:
# Step 3 — test the service in-process
from fastapi.testclient import TestClient
client = TestClient(app)

# Test /health
r = client.get("/health")
print(f"GET  /health        → {r.status_code} {r.json()}")
assert r.status_code == 200

# Test /draft with a known shipment
r = client.post("/draft", json={"email": "Where is my parcel PF-1003?", "shipment_id": "PF-1003"})
print(f"\nPOST /draft (PF-1003) → {r.status_code}")
print(f"  Draft: {r.json()['draft'][:100]}...")
assert r.status_code == 200
assert r.json()["shipment_id"] == "PF-1003"

# Test /draft with an ID extracted from the email
r = client.post("/draft", json={"email": "Where is my parcel PF-1007?"})
print(f"\nPOST /draft (extract) → {r.status_code}, found: {r.json()['shipment_id']}")
assert r.status_code == 200
assert r.json()["shipment_id"] == "PF-1007"

# Test /draft with an unknown ID
r = client.post("/draft", json={"email": "where is PF-9999?", "shipment_id": "PF-9999"})
print(f"\nPOST /draft (unknown) → {r.status_code} {r.json()['detail']}")
assert r.status_code == 404

# Test /draft with no ID at all
r = client.post("/draft", json={"email": "do you ship to vietnam?"})
print(f"\nPOST /draft (no ID)    → {r.status_code} {r.json()['detail']}")
assert r.status_code == 400

print("\n✅ All 5 service tests passed.")

### Step 4 — Add the 2 NEW endpoints that Phase 1 didn't have

`/draft` is the same drafter wrapped in HTTP. The new Phase 2 endpoints are the ones that make the service *better* than the CLI.

**`/retrieve` — the search box.** It finds the right policy chunks from the style guide for a query. The drafter uses this in Phase 2 T2 to ground its replies in the right policy.

**`/eval` — the report card.** It grades the drafter on a 30-row eval set and tells you how it's doing. Without this, every change is a guess.

In [ ]:
# Step 4 — add /retrieve and /eval
from pydantic import BaseModel
import time

# Simple token-overlap retriever (real Phase 2 T2 uses embeddings + BM25)
def tokenize(text: str) -> set[str]:
    return set(t.lower() for t in re.findall(r"[A-Za-z]+", text))

def chunk_style_guide(text: str) -> list[dict]:
    """Split the style guide by H2 sections."""
    chunks = []
    current = {"title": "intro", "body": ""}
    for line in text.split("\n"):
        if line.startswith("## "):
            if current["body"].strip():
                chunks.append(current)
            current = {"title": line[3:].strip(), "body": ""}
        else:
            current["body"] += line + "\n"
    if current["body"].strip():
        chunks.append(current)
    return chunks

chunks = chunk_style_guide(style_guide)
chunk_tokens = [(c, tokenize(c["title"] + " " + c["body"])) for c in chunks]

class RetrieveRequest(BaseModel):
    q: str = Field(..., min_length=1, max_length=500)
    k: int = Field(3, ge=1, le=10)

@app.post("/retrieve")
def retrieve(req: RetrieveRequest) -> dict:
    """Find the top-k most relevant style-guide chunks for a query."""
    q_tokens = tokenize(req.q)
    scored = []
    for chunk, c_tokens in chunk_tokens:
        overlap = len(q_tokens & c_tokens)
        if overlap > 0:
            scored.append((overlap, chunk))
    scored.sort(key=lambda x: -x[0])
    return {
        "query": req.q,
        "results": [
            {"title": c["title"], "score": float(s), "preview": c["body"][:200]}
            for s, c in scored[: req.k]
        ],
    }

# Test it
r = client.post("/retrieve", json={"q": "customs duty Vietnam", "k": 2})
print(f"POST /retrieve → {r.status_code}")
for hit in r.json()["results"]:
    print(f"  - {hit['title']!r} score={hit['score']}")
    print(f"    {hit['preview'][:100]}...")

print("\n✅ /retrieve works.")

### Step 5 — `/eval` — the drafter grades itself

**Why does a drafter need to grade itself?**

Because every time you change the prompt, the retriever, or the LLM, you might make it better OR worse. Without a grade, you're guessing. With a grade, you know.

**The 4 metrics, explained like you're 10:**

| Metric | What it measures | Simple explanation |
|---|---|---|
| **Faithfulness** | Did the drafter make up anything? | "Are all the facts in the draft actually in the lookup?" |
| **Answer relevance** | Did the draft answer the question? | "Does the draft talk about the same thing as the email?" |
| **Context precision** | Did we retrieve the right stuff? | "Are the top-3 chunks the ones the draft needed?" |
| **Context recall** | Did we miss anything important? | "Did we retrieve ALL the chunks the draft needed?" |

Each metric is a number from 0.0 to 1.0. 1.0 is perfect. 0.0 is terrible. **The eval set is the test, the metrics are the grades, the baseline is the high-water mark.**

In [ ]:
# Step 5 — /eval: a simplified eval harness
def jaccard(a: set, b: set) -> float:
    if not a and not b: return 1.0
    if not a or not b: return 0.0
    return len(a & b) / len(a | b)

def faithfulness(draft: str, shipment: dict) -> float:
    """Are all the facts in the draft actually in the shipment data?"""
    draft_tokens = tokenize(draft)
    fact_tokens = tokenize(shipment["status"] + " " + shipment["destination"])
    return jaccard(draft_tokens & fact_tokens, fact_tokens)

def answer_relevance(draft: str, email: str) -> float:
    """Does the draft talk about the same thing as the email?"""
    return jaccard(tokenize(draft), tokenize(email))

# A tiny eval set (real Phase 2 has 30 rows in shared/eval_set.jsonl)
eval_set = [
    {"email": "Where is my parcel PF-1003?", "shipment_id": "PF-1003"},
    {"email": "PF-1007 hasn't arrived.", "shipment_id": "PF-1007"},
    {"email": "PF-1012 status please", "shipment_id": "PF-1012"},
]

class EvalRequest(BaseModel):
    set_name: str = Field("default", description="Which eval set to run")

@app.post("/eval")
def eval_endpoint(req: EvalRequest) -> dict:
    """Run the eval set, return per-row and aggregate metrics."""
    rows = []
    for row in eval_set:
        r = client.post("/draft", json=row)
        d = r.json()
        shipment = shipments_db[row["shipment_id"]]
        rows.append({
            "shipment_id": row["shipment_id"],
            "faithfulness": faithfulness(d["draft"], shipment),
            "answer_relevance": answer_relevance(d["draft"], row["email"]),
        })
    agg = {
        "faithfulness":     sum(r["faithfulness"]     for r in rows) / len(rows),
        "answer_relevance": sum(r["answer_relevance"] for r in rows) / len(rows),
    }
    return {"rows": rows, "aggregate": agg, "n": len(rows)}

r = client.post("/eval", json={"set_name": "default"})
import json as _j
print(_j.dumps(r.json(), indent=2))

## 🏛️ FDE Lens — the industry standard for AI services in 2025

The 4-endpoint shape (`/health` + `/draft` + `/retrieve` + `/eval`) is **the de facto industry standard for a production RAG service in 2025.** Here's the map:

```
  PacificFreight      Industry standard              Open-source
  (this course)       (Anyscale, LangChain,         reference
                        LlamaIndex, Pinecone)
  ─────────────       ──────────────────────        ─────────────
  /health             liveness/readiness probe      same
  /draft              /invoke or /chat               same
  /retrieve           /search or /query             same
  /eval               /grade (or external: RAGAS)    same
  (Pydantic schemas)  Pydantic or JSON Schema        same
  (uvicorn)           gunicorn+uvicorn or vLLM       same
  (no auth in dev)    OAuth2 / API key in prod       same
```

**The tools the industry is using in 2025 (and how they fit):**

| Layer | Industry tools (2025) | What they replace | Why you might pick them |
|---|---|---|---|
| **LLM serving** | OpenAI, Anthropic, vLLM, Ollama, TGI | The `complete()` call in this notebook | vLLM if you have GPUs and want 10x cheaper; Ollama for local dev |
| **Embeddings** | OpenAI `text-embedding-3`, Cohere, BGE, sentence-transformers | The token-overlap retriever in this notebook | BGE if you need offline; OpenAI if you want zero setup |
| **Vector DB** | Pinecone, Weaviate, Qdrant, Chroma, pgvector | The in-memory chunk dict | Pinecone if you want zero ops; pgvector if you already have Postgres |
| **Orchestration** | LangChain, LlamaIndex, Haystack | The 5-piece pipeline | LangChain is most popular; LlamaIndex is best for RAG; Haystack for production |
| **Eval** | RAGAS, DeepEval, TruLens, Phoenix | The 4 metrics in this notebook | RAGAS is the standard; Phoenix for visual debugging |
| **Serving** | FastAPI, Flask, LitServe, Ray Serve | The FastAPI app in this notebook | FastAPI is the Python default; LitServe if you're serving LLMs at scale |
| **Observability** | LangSmith, Helicone, Phoenix, Langfuse, Datadog | The `print()` and counters in this notebook | LangSmith if you use LangChain; Phoenix if you want open-source |
| **Container** | Docker, Podman, Fly.io, Railway | The `Dockerfile` in `service/` | Docker is the default; Fly.io for easy deploys |

**The pattern:** every AI service in production is the same shape. The pieces are:
1. An HTTP server (FastAPI, Flask, LitServe)
2. A retriever (BM25, dense, hybrid)
3. An LLM call (OpenAI, vLLM, Ollama)
4. An eval harness (RAGAS, custom metrics)
5. Observability (LangSmith, Phoenix, Datadog)
6. A container (Docker, Podman)

**The benefit of the standard:** when you learn this shape, you can read ANY AI service codebase in the wild. They're all the same 6 pieces, with different brand names bolted on.

### The 2025 industry moves you should know about

1. **Agents & MCP** — every vendor now ships an "agent" that calls tools. The drafter becomes a tool-user, not just a chat-replier. (Phase 4 Project 1.)
2. **Hybrid retrieval** — BM25 + dense + RRF is the new default. The mock store from this notebook is dead. (Phase 3 T1.)
3. **Streaming** — every chat UX is now streaming. `/draft/stream` is the production shape. (Phase 3 T2.)
4. **Small models (SLMs)** — Qwen 1.5B, Phi-3, Gemma 2B are now good enough for narrow tasks. The cost ceiling is 1/20th of GPT-4o. (Phase 4 Project 3.)
5. **Eval-in-CI** — every prompt change runs the eval suite. If metrics drop > 5%, the PR doesn't merge. (Phase 3 T2.)
6. **Open weights** — Llama 3.1, Mistral, Qwen 2.5 are good enough for 80% of use cases. No data leaves your infra. (Phase 4 Project 3.)
7. **Structured output** — every LLM call now returns JSON, not free text. Pydantic validates it. (This notebook's `DraftResponse`.)
8. **Prompt caching** — Anthropic, OpenAI now cache repeated system prompts. 50% cost reduction for free. (Phase 3 T3.)

**Your takeaway:** the 5-piece drafter from Phase 1 is the *core* of every one of these 2025 industry moves. When the industry adds an agent, you wrap your drafter in an agent. When it adds streaming, you wrap your drafter in SSE. **The 5 pieces are the foundation. The 2025 stack is the furniture on top.**

## 🧪 Test the FULL Phase 2 service end-to-end

Let's run the 5-test mini-suite that mirrors what the real Phase 2 test suite does (the real one has 13 cases in `service/tests/test_app.py`).

In [ ]:
# Full end-to-end test suite (mirrors the real Phase 2 test_app.py)
tests = []

# 1. Health
r = client.get("/health")
tests.append(("test_health", r.status_code == 200 and r.json()["status"] == "ok"))

# 2. Draft with known shipment
r = client.post("/draft", json={"email": "PF-1003?", "shipment_id": "PF-1003"})
tests.append(("test_draft_known", r.status_code == 200 and "PF-1003" in r.json()["draft"]))

# 3. Draft with ID extracted from email
r = client.post("/draft", json={"email": "where is PF-1007?"})
tests.append(("test_draft_extract", r.status_code == 200 and r.json()["shipment_id"] == "PF-1007"))

# 4. Draft with no ID
r = client.post("/draft", json={"email": "do you ship to vietnam?"})
tests.append(("test_draft_no_id", r.status_code == 400))

# 5. Retrieve returns chunks
r = client.post("/retrieve", json={"q": "customs", "k": 2})
tests.append(("test_retrieve", r.status_code == 200 and len(r.json()["results"]) > 0))

for name, passed in tests:
    mark = "✅" if passed else "❌"
    print(f"  {mark}  {name}")
passed = sum(1 for _, p in tests if p)
print(f"\n{passed}/{len(tests)} passed")
assert passed == len(tests), "All tests must pass!"

## 🌙 Reflect

**Before you move on, write down 3 things:**

1. The 4 endpoints of the Phase 2 service (`/health`, `/draft`, `/retrieve`, `/eval`). What each one does.
2. The 2 things Pydantic does for us (define the schema + validate the input). Why this matters in production.
3. The 4 RAGAS-style metrics (faithfulness, answer relevance, context precision, context recall). What each one measures.

**Then answer the FDE Lens question:** if a customer said "we want to use Pinecone, not pgvector," which endpoint would change? Which would stay the same? (Hint: only `/retrieve` changes. The 5-piece pipeline doesn't care where the chunks live.)

## 🎓 What's next

Phase 2's drafter works in mock mode and has 4 endpoints. But:

- The retriever is a toy (token overlap). It misses semantically-similar queries.
- There's no streaming (Mei waits 1.8s for the full draft).
- There's no feedback (we don't know which drafts Mei liked).
- There's no circuit breaker (when OpenAI goes down, the service hangs).
- There's no runbook (nobody knows what to do at 2am).

**Phase 3 fixes all of these.** The drafter gets a hybrid retriever, SSE streaming, a `/feedback` endpoint, a circuit breaker, a rate limiter, a redactor, a runbook, a RACI, and an on-call rotation.

Go to [Phase 3 — Deployment](../../phase-3-deployment/) to see the drafter get its safety gear.